# PV Project_Disproportionality Analysis

### Importing Libraries

In [78]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
import warnings
from scipy.special import gammaln, digamma, expit, logit
from scipy.optimize import minimize
from scipy.stats import gamma as G

### Establish DB connection

In [79]:
# Establishing connection wit the database

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DB_PATH = PROJECT_ROOT / "data" / "faers_db.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=True) # read_only=True for exploration, and read_only=False only when creating tables.

### Identify Tables

In [80]:
pd.set_option("display.max_columns", 60, "display.width", 220, "display.max_colwidth", 50)
print(con.sql("SHOW TABLES").df())

                     name
0               case_base
1               case_drug
2           case_exposure
3              case_month
4            case_outcome
5                 case_pt
6             case_source
7              clean_demo
8                 demo_us
9             drug_detail
10               exposure
11             faers_flat
12                    m_N
13                   m_ev
14                  m_grp
15                 m_pair
16                 months
17            pair_counts
18              pair_grid
19             pair_month
20                ps_case
21          ps_case_event
22                ps_drug
23               raw_demo
24               raw_indi
25               raw_reac
26               raw_ther
27               ref_drug
28       ref_known_signal
29          signal_labels
30              signal_ts
31       study_case_event
32            study_cases
33        study_drug_rows
34   v_ingredient_summary
35   v_known_signal_check
36  v_known_signal_family
37        v_

### Explore the selected tables

In [81]:
pair_counts = con.sql("SELECT * FROM pair_counts").df()
pair_month= con.sql ("SELECT * FROM pair_month").df()
pair_grid= con.sql ("SELECT * FROM pair_grid").df()
v_signals= con.sql ("SELECT * FROM v_signals").df()

In [82]:
pair_counts.describe()
pair_counts

,ingredient,pt,a,n_exp,n_excl,N,n_event_all,n_event_excl
0,liraglutide,PANCREATIC ENLARGEMENT,1,11604,7605,5374222,55,1
1,empagliflozin,BENIGN PROSTATIC HYPERPLASIA,2,6407,3685,5374222,1130,2
2,dapagliflozin,FRACTURE,3,4851,2414,5374222,2490,2
3,dapagliflozin,MUSCULOSKELETAL STIFFNESS,8,4851,2414,5374222,21027,3
4,empagliflozin,GALLBLADDER DISORDER,2,6407,3685,5374222,3312,7
...,...,...,...,...,...,...,...,...
9246,canagliflozin,PAROTID DUCT OBSTRUCTION,2,18865,4520,5374222,7,0
9247,canagliflozin,LYMPHOEDEMA,3,18865,4520,5374222,1312,0
9248,canagliflozin,URINE ALCOHOL TEST POSITIVE,5,18865,4520,5374222,95,0
9249,canagliflozin,ATRIOVENTRICULAR BLOCK,1,18865,4520,5374222,1109,0


In [ ]:
v_signals

## Classical disproportionality analysis

* pair_counts table (9251 drug-event pairs, each against the full US background). 
*  Can be used as a proper signal table with four methods, a noise filter, seriousness, validation against the known signals, and sensitivity runs. 

#### 1. Pull the counts and compute PRR, ROR and IC

In [83]:

def disproportionality(pair_counts):
    df = pair_counts.copy()
    df["b"] = df.n_exp - df.a
    df["c"] = df.n_event_all - df.a - df.n_event_excl
    df["d"] = (df.N - df.n_exp - df.n_excl) - df.c
    a, b, c, d = [df[x].astype(float) for x in "abcd"]
    ok = (b > 0) & (c > 0) & (d > 0)                    # undefined otherwise; flagged, not dropped
    n = a + b + c + d
    df["E"] = (a + b) * (a + c) / n                      # expected count under independence

    with np.errstate(all="ignore"):
        df["PRR"]    = np.where(ok, (a/(a+b)) / (c/(c+d)), np.nan)
        se_prr       = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
        df["PRR_lo"] = np.exp(np.log(df.PRR) - 1.96*se_prr)
        df["chi2"]   = np.where(ok, (a*d - b*c)**2 * n / ((a+b)*(c+d)*(a+c)*(b+d)), np.nan)
        df["ROR"]    = np.where(ok, (a*d)/(b*c), np.nan)
        df["ROR_lo"] = np.exp(np.log(df.ROR) - 1.96*np.sqrt(1/a + 1/b + 1/c + 1/d))
        df["IC"]     = np.log2((a + 0.5) / (df.E + 0.5))
        df["IC025"]  = df.IC - 3.3*(a + 0.5)**-0.5 - 2*(a + 0.5)**-1.5   # Norén approximation
    return df

res = disproportionality(pair_counts)
print(len(res), "pairs;", res.PRR.isna().sum(), "with an empty cell (metrics undefined)")


9251 pairs; 25 with an empty cell (metrics undefined)


In [84]:
res

,ingredient,pt,a,n_exp,n_excl,N,n_event_all,n_event_excl,b,c,d,E,PRR,PRR_lo,chi2,ROR,ROR_lo,IC,IC025
0,liraglutide,PANCREATIC ENLARGEMENT,1,11604,7605,5374222,55,1,11603,53,5354960,0.116762,8.707168,1.204213,6.695750,8.707833,1.204103,1.282177,-2.500924
1,empagliflozin,BENIGN PROSTATIC HYPERPLASIA,2,6407,3685,5374222,1130,2,6405,1126,5363004,1.345693,1.487086,0.371530,0.318586,1.487238,0.371408,0.437765,-2.155302
2,dapagliflozin,FRACTURE,3,4851,2414,5374222,2490,2,4848,2485,5364472,2.246783,1.335647,0.430622,0.252855,1.335855,0.430388,0.349612,-1.719754
3,dapagliflozin,MUSCULOSKELETAL STIFFNESS,8,4851,2414,5374222,21027,3,4843,21016,5345941,18.985679,0.421150,0.210706,6.387385,0.420194,0.209987,-1.196879,-2.409475
4,empagliflozin,GALLBLADDER DISORDER,2,6407,3685,5374222,3312,7,6405,3303,5360827,3.942834,0.506951,0.126758,0.959066,0.506797,0.126665,-0.829552,-3.422620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9246,canagliflozin,PAROTID DUCT OBSTRUCTION,2,18865,4520,5374222,7,0,18863,5,5350832,0.024593,113.455330,22.012662,159.234699,113.467253,22.012232,2.252659,-0.340409
9247,canagliflozin,LYMPHOEDEMA,3,18865,4520,5374222,1312,0,18862,1309,5349528,4.609358,0.650050,0.209398,0.564027,0.649994,0.209342,-0.545787,-2.615153
9248,canagliflozin,URINE ALCOHOL TEST POSITIVE,5,18865,4520,5374222,95,0,18860,90,5350747,0.333757,15.757685,6.403750,65.469744,15.761597,6.403887,2.721733,1.159553
9249,canagliflozin,ATRIOVENTRICULAR BLOCK,1,18865,4520,5374222,1109,0,18864,1108,5349729,3.896172,0.255991,0.036029,2.160871,0.255952,0.036019,-1.551285,-5.334386


#### 2: EBGM prior, fitted on the full ingredient × event grid  (DuMouchel’s shrinkage)

EBGM shrinks small-count estimates toward 1. It fits a two-gamma prior to all observed pairs, then computes the posterior mean of the log relative reporting rate (EBGM) and its 5th percentile (EB05). Because only pairs with a ≥ 1 exist in your table, I fit a zero-truncated likelihood, so the prior isn’t biased by the missing zero cells.


In [85]:
# Every ingredient x PT cell, including zero cells
grid = con.sql("""
WITH ev_tot AS (SELECT pt, COUNT(*) AS n_e FROM case_pt GROUP BY pt),
grp_n  AS (SELECT ingredient, g, COUNT(*) AS n FROM exposure GROUP BY 1, 2),
ev_grp AS (SELECT x.ingredient, x.g, p.pt, COUNT(*) AS n
           FROM exposure x JOIN case_pt p USING (caseid) GROUP BY 1, 2, 3),
ings AS (SELECT ingredient FROM grp_n WHERE g = 'exp'),
tot  AS (SELECT COUNT(DISTINCT caseid) AS N FROM case_pt)
SELECT i.ingredient, t.pt,
       COALESCE(a.n, 0)  AS a,
       ne.n              AS n_exp,
       COALESCE(nx.n, 0) AS n_excl,
       (SELECT N FROM tot) AS N,
       t.n_e             AS n_event_all,
       COALESCE(xe.n, 0) AS n_event_excl
FROM ings i CROSS JOIN ev_tot t
JOIN grp_n ne ON ne.ingredient = i.ingredient AND ne.g = 'exp'
LEFT JOIN grp_n nx ON nx.ingredient = i.ingredient AND nx.g = 'excl'
LEFT JOIN ev_grp a  ON a.ingredient  = i.ingredient AND a.pt  = t.pt AND a.g  = 'exp'
LEFT JOIN ev_grp xe ON xe.ingredient = i.ingredient AND xe.pt = t.pt AND xe.g = 'excl'
""").df()

grid["E"] = grid.n_exp * (grid.n_event_all - grid.n_event_excl) / (grid.N - grid.n_excl)
print("cells:", len(grid), "| observed:", int((grid.a > 0).sum()),
      "| E<=0 or NaN:", int((~(grid.E > 0)).sum()))

g = grid[grid.E > 0]
n_fit, E_fit = g.a.values.astype(float), g.E.values

def _nb_log(n, E, al, be):
    p = be / (be + E)
    return gammaln(al + n) - gammaln(al) - gammaln(n + 1) + al*np.log(p) + n*np.log1p(-p)

def nll_safe(th, n, E):
    a1, b1, a2, b2 = np.exp(th[:4]); w = expit(th[4])
    l = np.logaddexp(np.log(w) + _nb_log(n, E, a1, b1),
                     np.log1p(-w) + _nb_log(n, E, a2, b2))
    v = -l.sum()
    return v if np.isfinite(v) else 1e12

# order: log a1, log b1, log a2, log b2, logit w   (b1 may go very small)
bounds = [(-3, 3), (-8, 3), (-3, 3), (-3, 3), (-6, 6)]
starts = [np.r_[np.log([0.2, 0.1, 2.0, 4.0]), logit(1/3)],
          np.r_[np.log([0.5, 0.5, 1.5, 2.0]), 0.0],
          np.r_[np.log([0.1, 0.05, 3.0, 3.0]), logit(0.7)],
          np.r_[np.log([0.118, 0.0284, 0.793, 2.110]), logit(0.1355)]]   # your earlier solution

fits = [minimize(nll_safe, s, args=(n_fit, E_fit), method="L-BFGS-B", bounds=bounds) for s in starts]
for f in fits: print(round(f.fun, 1), f.success)
fit = min(fits, key=lambda r: r.fun)

a1, b1, a2, b2 = np.exp(fit.x[:4]); w = expit(fit.x[4])
prior = dict(alpha1=a1, beta1=b1, alpha2=a2, beta2=b2, w=w, nll=fit.fun, ok=fit.success)
print({k: round(float(v), 4) if not isinstance(v, (bool, np.bool_)) else bool(v) for k, v in prior.items()})
at_bound = [nm for nm, v, (lo, hi) in zip(["a1","b1","a2","b2","w"], fit.x, bounds)
            if abs(v - lo) < 1e-3 or abs(v - hi) < 1e-3]
print("parameters at a bound:", at_bound or "none")

cells: 119000 | observed: 9251 | E<=0 or NaN: 14
38967.3 True
38967.3 True
38967.3 True
38967.3 True
{'alpha1': 0.1184, 'beta1': 0.0284, 'alpha2': 0.7926, 'beta2': 2.11, 'w': 0.1355, 'nll': 38967.333, 'ok': True}
parameters at a bound: none


In [86]:
# Apply the prior (EBGM and EB05)

def ebgm_apply(df, a1, b1, a2, b2, w):
    n, E = df.a.values.astype(float), df.E.values
    valid = E > 0
    nn, EE = n[valid], E[valid]
    l1 = np.log(w)    + _nb_log(nn, EE, a1, b1)
    l2 = np.log1p(-w) + _nb_log(nn, EE, a2, b2)
    Q  = np.exp(l1 - np.logaddexp(l1, l2))
    ebgm_v = np.exp(Q*(digamma(a1 + nn) - np.log(b1 + EE)) + (1 - Q)*(digamma(a2 + nn) - np.log(b2 + EE)))
    lo, hi = np.zeros_like(nn), np.full_like(nn, 1e3)
    for _ in range(60):                       # bisection for the 5th percentile
        mid = (lo + hi) / 2
        cdf = Q*G.cdf(mid, a1 + nn, scale=1/(b1 + EE)) + (1 - Q)*G.cdf(mid, a2 + nn, scale=1/(b2 + EE))
        lo, hi = np.where(cdf < 0.05, mid, lo), np.where(cdf < 0.05, hi, mid)
    ebgm = np.full(len(df), np.nan); eb05 = np.full(len(df), np.nan)
    ebgm[valid], eb05[valid] = ebgm_v, (lo + hi) / 2
    return ebgm, eb05

res["EBGM"], res["EB05"] = ebgm_apply(res, a1, b1, a2, b2, w)
res["raw_rr"] = res.a / res.E

# Check: shrinkage should be strong at small a and fade as a grows
chk = res.dropna(subset=["EBGM", "raw_rr"]).copy()
chk["ratio"] = chk.EBGM / chk.raw_rr
chk["a_bin"] = pd.cut(chk.a, [0, 2, 4, 9, 29, 1e9])
print(chk.groupby("a_bin", observed=True).ratio.median().round(2))



a_bin
(0.0, 2.0]              0.41
(2.0, 4.0]              0.75
(4.0, 9.0]              0.86
(9.0, 29.0]             0.94
(29.0, 1000000000.0]    0.99
Name: ratio, dtype: float64


#### 3: Signal criteria and noise filter

In [87]:

NOISE = (r"DRUG INEFFECTIVE|PRODUCT USE|OFF LABEL|INCORRECT (?:DOSE|DRUG|ROUTE)|WRONG|"
         r"DEVICE|PRODUCT (?:QUALITY|CONTAMINATION|PACKAGING|STORAGE|COUNTERFEIT)|"
         r"INTENTIONAL|ACCIDENTAL|MEDICATION ERROR|NO ADVERSE EVENT|INJECTION SITE|INTERCEPTED PRODUCT PRESCRIBING ERROR|PRODUCT DOSAGE FORM CONFUSION|"
         r"EXPIRED|DRUG DOSE OMISSION|UNEVALUABLE|TREATMENT NONCOMPLIANCE|UNDERDOSE|OVERDOSE|PRODUCT PRESCRIBING ERROR|PRODUCT LABEL CONFUSION|"
         r"INTERCEPTED PRODUCT DISPENSING ERROR|INTERCEPTED DRUG DISPENSING ERROR|PRODUCT DISPENSING ERROR|PRODUCT COMMUNICATION ISSUE|"
         r"DRUG DOSE TITRATION NOT PERFORMED|DRUG TITRATION ERROR|PRODUCT DOSE OMISSION|PRODUCT ADMINISTERED AT INAPPROPRIATE SITE|"
         r"NEEDLE TRACK MARKS|WEIGHT LOSS POOR|GLYCOSYLATED|INJECTION SITE|INAPPROPRIATE SCHEDULE OF DRUG ADMINISTRATION|"
         r"DRUG INTERACTION|LACK OF EFFICACY|BLOOD GLUCOSE (?:INCREASED|ABNORMAL|DECREASED|FLUCTUATION)")
res["is_noise"] = res.pt.str.contains(NOISE, regex=True, na=False)
print(int(res.is_noise.sum()), "noise pairs of", len(res))

def flag(df, min_a=3):
    df = df.copy()
    df["evans"]  = (df.a >= min_a) & (df.PRR >= 2) & (df.chi2 >= 4)          # PRR criterion
    df["ror_sig"] = (df.a >= min_a) & (df.ROR_lo > 1)
    df["ic_sig"]  = (df.a >= min_a) & (df.IC025 > 0)
    df["eb_sig"]  = (df.a >= min_a) & (df.EB05 >= 2)                         # DuMouchel's usual cut
    df["n_methods"] = df[["evans","ror_sig","ic_sig","eb_sig"]].sum(axis=1)
    df["signal"]  = df.evans & df.ror_sig & df.ic_sig                        # same rule as signal_ts
    df["strong"]  = df.signal & df.eb_sig
    return df   
    

sig = flag(res, 3)
cols = ["evans", "ror_sig", "ic_sig", "eb_sig", "signal", "strong"]
print(sig.groupby("ingredient")[cols].sum())
print("\nafter removing noise PTs:")
print(sig[~sig.is_noise].groupby("ingredient")[["signal", "strong"]].sum())

# Small-count pairs: shrinkage should cut the number passing
small = sig[(sig.a <= 4) & ~sig.is_noise]
print(f"\n{len(small)} pairs with a<=4 | raw RR>=2: {(small.a/small.E >= 2).sum()} | EB05>=2: {(small.EB05 >= 2).sum()}")

  

511 noise pairs of 9251
               evans  ror_sig  ic_sig  eb_sig  signal  strong
ingredient                                                   
canagliflozin    284      302     268     197     251     197
dapagliflozin    199      213     183     123     172     123
dulaglutide      156      176     145      96     132      96
empagliflozin    183      193     167     120     162     120
ertugliflozin     21       23      22      14      21      14
liraglutide      160      186     152      96     136      96
semaglutide       98      107      85      59      81      59

after removing noise PTs:
               signal  strong
ingredient                   
canagliflozin     243     191
dapagliflozin     161     114
dulaglutide        84      59
empagliflozin     154     113
ertugliflozin      15      11
liraglutide       112      80
semaglutide        66      45

6328 pairs with a<=4 | raw RR>=2: 2443 | EB05>=2: 112


The “blood glucose increased” and “drug interaction” terms considered as  noise for this analysis.


In [88]:
# noise 
print(sig[sig.is_noise & sig.signal].sort_values("a", ascending=False)[["ingredient","pt","a"]].head(40))

         ingredient                                             pt     a
2764    dulaglutide                        BLOOD GLUCOSE INCREASED  2930
2408    dulaglutide                            INJECTION SITE PAIN  2928
3666    dulaglutide                             DRUG DOSE OMISSION  1298
492     dulaglutide                     INJECTION SITE HAEMORRHAGE  1135
483     dulaglutide                    INCORRECT DOSE ADMINISTERED  1008
2754    liraglutide                        BLOOD GLUCOSE INCREASED   858
4866    dulaglutide                           ACCIDENTAL UNDERDOSE   709
4637    dulaglutide                                      UNDERDOSE   694
488     dulaglutide                        INJECTION SITE BRUISING   649
2187    dulaglutide             GLYCOSYLATED HAEMOGLOBIN INCREASED   621
4180  canagliflozin                        BLOOD GLUCOSE INCREASED   602
467     dulaglutide                        BLOOD GLUCOSE DECREASED   567
4210    dulaglutide                        INJECTIO

In [89]:
sig["a_band"] = np.where(sig.a >= 5, "a>=5", "a 3-4")
sig.loc[sig.a < 3, "a_band"] = "a<3"

sig["tier"] = np.select(
    [sig.is_noise, sig.strong, sig.signal],
    ["0 Noise (excluded)", "1 Strong", "2 Signal, not strong"],
    default="3 Not flagged")

tab = pd.crosstab(sig.tier, sig.a_band, margins=True)
assert tab.loc["All", "All"] == len(sig)
print(tab)
print(pd.crosstab(sig[~sig.is_noise].tier, sig[~sig.is_noise].ingredient, margins=True))

a_band                a 3-4   a<3  a>=5   All
tier                                         
0 Noise (excluded)       64   183   264   511
1 Strong                 84     0   529   613
2 Signal, not strong     44     0   178   222
3 Not flagged          1002  5198  1705  7905
All                    1194  5381  2676  9251
ingredient            canagliflozin  dapagliflozin  dulaglutide  empagliflozin  ertugliflozin  liraglutide  semaglutide   All
tier                                                                                                                         
1 Strong                        191            114           59            113             11           80           45   613
2 Signal, not strong             52             47           25             41              4           32           21   222
3 Not flagged                  1628           1092         1634           1222            183         1486          660  7905
All                            1871           12

#### Summary (Drug-AR pairs --> Signal)
Of 9,251 drug-event pairs, 511 were removed as non-safety terms (medication error, product, efficacy/laboratory), leaving 8,740. A pair was a signal if it had ≥ 3 reports and met the Evans, ROR and IC025 criteria (n = 835), and a strong signal if it also had EB05 ≥ 2 (n = 613; 84 with 3-4 reports). A further 222 pairs met the frequentist criteria but not EB05 ≥ 2.

#### 4: Seriousness and the ranked signal list

Seriousness comes from the existing view and is case-level & not event level

In [90]:
ser = con.sql("SELECT * FROM v_pair_serious").df()
lab = con.sql("SELECT ingredient, pt, onset, status FROM signal_labels").df()

out = (sig.merge(ser[["ingredient", "pt", "serious_cases", "deaths", "hospitalised", "life_threat"]],
                 on=["ingredient", "pt"], how="left")
          .merge(lab, on=["ingredient", "pt"], how="left"))
out["pct_serious"] = (100 * out.serious_cases / out.a).round(1)

# 1. Noise-flagged signals: check what the filter removes
print("=== Noise-flagged signals (top 40) ===")
display(out[out.is_noise & out.signal].sort_values("a", ascending=False)
        [["ingredient", "pt", "a"]].head(40))

# 2. Weak tier: signals rejected by EB05
weak = out[out.signal & ~out.eb_sig & ~out.is_noise]
print(f"\n=== Weak tier: {len(weak)} signals, median a = {weak.a.median()} ===")
display(weak.sort_values("a", ascending=False)[["ingredient", "pt", "a", "PRR", "IC025", "EB05"]].head(20).round(2))

# 3. Strong signals per drug
top = out[out.strong & ~out.is_noise].sort_values(["ingredient", "EB05"], ascending=[True, False])
show = ["pt", "a", "PRR", "ROR_lo", "IC025", "EBGM", "EB05", "pct_serious", "deaths", "status"]
for ing, d in top.groupby("ingredient"):
    print(f"\n=== {ing}: {len(d)} strong signals (non-noise) ===")
    display(d[show].head(15).round(2))

=== Noise-flagged signals (top 40) ===


,ingredient,pt,a
2764,dulaglutide,BLOOD GLUCOSE INCREASED,2930
2408,dulaglutide,INJECTION SITE PAIN,2928
3666,dulaglutide,DRUG DOSE OMISSION,1298
492,dulaglutide,INJECTION SITE HAEMORRHAGE,1135
483,dulaglutide,INCORRECT DOSE ADMINISTERED,1008
2754,liraglutide,BLOOD GLUCOSE INCREASED,858
4866,dulaglutide,ACCIDENTAL UNDERDOSE,709
4637,dulaglutide,UNDERDOSE,694
488,dulaglutide,INJECTION SITE BRUISING,649
2187,dulaglutide,GLYCOSYLATED HAEMOGLOBIN INCREASED,621



=== Weak tier: 222 signals, median a = 8.5 ===


,ingredient,pt,a,PRR,IC025,EB05
1802,liraglutide,WEIGHT INCREASED,265,2.07,0.84,1.84
4249,liraglutide,HEART RATE INCREASED,110,2.23,0.83,1.84
4202,canagliflozin,RENAL INJURY,87,2.13,0.72,1.71
3667,canagliflozin,NEPHROLITHIASIS,84,2.39,0.88,1.91
3789,canagliflozin,CYSTITIS,66,2.33,0.79,1.79
1625,dapagliflozin,RENAL FAILURE,63,2.26,0.74,1.73
5169,canagliflozin,HAEMATURIA,58,2.19,0.67,1.65
3370,canagliflozin,URINARY INCONTINENCE,52,2.36,0.75,1.74
2840,dulaglutide,INCREASED APPETITE,47,2.32,0.70,1.68
2446,dapagliflozin,DRY MOUTH,46,2.35,0.72,1.70



=== canagliflozin: 191 strong signals (non-noise) ===


,pt,a,PRR,ROR_lo,IC025,EBGM,EB05,pct_serious,deaths,status
2179,LIMB AMPUTATION,233,6608.77,3552.65,7.21,263.70,236.46,85.0,3,emerging
1816,DIABETIC FOOT INFECTION,688,1383.99,1197.61,7.53,233.81,219.51,91.4,6,emerging
3806,OSTEOMYELITIS ACUTE,231,2047.51,1431.84,7.13,242.15,217.03,84.0,1,emerging
288,METATARSAL EXCISION,99,9360.06,2983.21,6.52,254.99,215.46,83.8,0,emerging
4546,TOE AMPUTATION,1915,763.95,779.01,7.55,206.89,199.23,75.5,15,emerging
3264,FOOT AMPUTATION,517,819.22,710.18,7.31,208.85,194.17,85.3,7,emerging
2512,GAS GANGRENE,95,1496.98,908.87,6.39,222.42,187.27,97.9,0,emerging
3268,DIABETIC FOOT,683,466.81,428.32,7.17,175.66,164.88,85.7,6,emerging
2493,AMPUTATION,285,495.93,415.14,6.91,177.63,160.97,60.4,4,emerging
2415,OSTEOMYELITIS CHRONIC,95,626.64,439.17,6.26,184.36,155.22,83.2,2,emerging


... (8 outputs left out)


=== liraglutide: 80 strong signals (non-noise) ===


,pt,a,PRR,ROR_lo,IC025,EBGM,EB05,pct_serious,deaths,status
2911,MEDULLARY THYROID CANCER,19,132.85,79.86,4.06,87.75,59.09,36.8,1,pre-existing
2880,PANCREATIC CARCINOMA STAGE IV,16,121.04,69.87,3.79,80.15,51.97,87.5,9,pre-existing
4533,PANCREATIC CARCINOMA,349,59.47,54.74,5.44,52.52,48.05,69.1,203,pre-existing
4910,PAPILLARY THYROID CANCER,44,73.04,53.32,4.72,60.19,46.63,36.4,0,pre-existing
128,PANCREATITIS,1109,53.83,55.67,5.46,48.24,45.90,41.9,19,pre-existing
1462,PANCREATIC CARCINOMA METASTATIC,68,43.34,33.98,4.55,38.84,31.67,79.4,44,pre-existing
4572,PANCREATITIS ACUTE,169,36.72,31.81,4.70,33.82,29.74,72.8,6,pre-existing
3728,ERUCTATION,294,32.46,29.52,4.66,30.26,27.47,3.1,0,emerging
2199,ADENOCARCINOMA PANCREAS,28,42.79,29.11,3.92,37.23,26.96,75.0,8,pre-existing
4897,LIPASE INCREASED,75,35.39,28.14,4.38,32.37,26.66,17.3,0,pre-existing



=== semaglutide: 45 strong signals (non-noise) ===


,pt,a,PRR,ROR_lo,IC025,EBGM,EB05,pct_serious,deaths,status
7314,RETINOPATHY,19,85.79,54.66,3.97,72.30,48.69,5.3,0,emerging
1196,ERUCTATION,68,37.73,30.51,4.48,36.40,29.68,4.4,0,emerging
5268,DIABETIC RETINOPATHY,11,46.42,25.65,2.93,39.38,23.20,0.0,0,emerging
1687,PANCREATITIS,87,20.11,16.85,3.81,19.74,16.49,31.0,1,emerging
3000,DIABETES MELLITUS INADEQUATE CONTROL,18,15.39,9.74,2.67,14.63,9.74,5.6,0,emerging
5046,PANCREATITIS ACUTE,15,15.87,9.60,2.55,14.93,9.54,73.3,0,emerging
8269,CHOLECYSTITIS ACUTE,5,25.48,10.57,1.42,20.36,8.95,80.0,0,emerging
1751,EUGLYCAEMIC DIABETIC KETOACIDOSIS,3,47.54,15.17,0.56,27.77,8.89,100.0,0,emerging
9041,VOMITING PROJECTILE,6,21.17,9.49,1.63,17.88,8.53,0.0,0,emerging
7617,PAPILLARY THYROID CANCER,4,29.68,11.08,1.06,21.83,8.50,0.0,0,emerging


#### Review the watchlist pairs

In [91]:
# 178 watchlist pairs (not strong signal, a>=5)
watch = sig[(sig.tier == "2 Signal, not strong") & (sig.a >= 5)]
watch.sort_values("EB05", ascending=False)[["ingredient","pt","a","PRR","ROR_lo","IC025","EBGM","EB05"]].head(40)

,ingredient,pt,a,PRR,ROR_lo,IC025,EBGM,EB05
2480,dapagliflozin,SEPSIS,46,2.765752,2.080648,0.948062,2.599023,1.999920
3026,canagliflozin,DIZZINESS POSTURAL,25,3.242322,2.187400,0.955288,2.921262,1.996801
1600,empagliflozin,LACTIC ACIDOSIS,13,4.166916,2.418654,0.955815,3.608567,1.989610
6513,liraglutide,SENSATION OF FOREIGN BODY,15,3.897634,2.346213,0.954730,3.397550,1.984624
1284,dapagliflozin,NEPHROLITHIASIS,28,3.090702,2.138966,0.942095,2.818626,1.979761
454,semaglutide,PRURITUS GENERALISED,14,3.983626,2.365680,0.948424,3.469656,1.973467
4978,dapagliflozin,BLOOD CREATINE PHOSPHOKINASE INCREASED,9,5.080229,2.641668,0.921130,4.267527,1.960799
7824,dulaglutide,PAPILLARY THYROID CANCER,8,5.596275,2.775621,0.905397,4.577167,1.959807
216,ertugliflozin,PALPITATIONS,9,5.060494,2.664808,0.920585,4.264795,1.959513
819,canagliflozin,ORAL CANDIDIASIS,24,3.198928,2.140638,0.920653,2.863471,1.941286


#### Check the 28 pairs with a < 3 and EB05 ≥ 2:

All four flags in our flag() function require a >= min_a, and min_a is 3. A pair with 1 or 2 reports therefore gets eb_sig = False, even if its EB05 is above 2. "signal" and "strong" depend on those flags, so they are False too. The crosstab confirms it: the a<3 column has 0 in both Strong and Signal-not-strong, and 5,220 in Not flagged.
 The 112 pairs with EB05 >=2 (from "small" of  Small-count pairs)& 84 of these are in Strong tier(a=3-4). So the rest (112-84)29 with a =1-2 & EB05>=2 can be signals too but was not flagged , hence checking them too.

In [92]:
x = sig[~sig.is_noise & (sig.a < 3) & (sig.EB05 >= 2)]
print(len(x), x.tier.unique())  
x[["ingredient","pt","a","E","EBGM","EB05"]].sort_values("EB05", ascending=False)

28 ['3 Not flagged']


,ingredient,pt,a,E,EBGM,EB05
7027,empagliflozin,MYCOPLASMA GENITALIUM INFECTION,2,0.002386,50.436655,11.016723
8808,empagliflozin,PERINEAL NECROSIS,2,0.002386,50.436655,11.016723
9133,empagliflozin,MALE REPRODUCTIVE TRACT DISORDER,2,0.003579,48.363445,10.426781
6920,ertugliflozin,GENITAL INFECTION,2,0.003730,48.110892,10.354314
3942,dapagliflozin,CYSTITIS GLANDULARIS,2,0.004515,46.838716,9.987095
770,empagliflozin,PENILE ODOUR,2,0.007158,42.956180,8.839878
7781,ertugliflozin,GENITAL DISCOMFORT,2,0.009459,40.004032,7.932946
6629,ertugliflozin,EUGLYCAEMIC DIABETIC KETOACIDOSIS,2,0.010592,38.675500,7.511905
6351,empagliflozin,GENITAL DISCOLOURATION,2,0.010737,38.510696,7.459016
7029,empagliflozin,URINE VISCOSITY ABNORMAL,2,0.011930,37.199018,7.032243


In [93]:
print(x.groupby("ingredient").size())   # which drugs they belong to

ingredient
canagliflozin     4
dapagliflozin     4
dulaglutide       2
empagliflozin    11
ertugliflozin     5
semaglutide       2
dtype: int64


#### Why EB05 is high with 1-2 reports: 
* it means E is tiny, so even one or two reports give a large ratio. Pairs with E (E column values) well below 0.1 are most likely small-denominator artifacts.
* Two possible kinds:
    * rare, serious, designated-medical-event-type terms (for example, Fournier’s-like or hepatic events), which deserve a manual look.
    * odd or very specific PTs with no clinical meaning, which  can be ignored.(Found "PRODUCT DOSAGE FORM CONFUSION" which was moved to Noise)

#### 5: Validate against the known signals

In [94]:
known = [("canagliflozin", "AMPUTATION"), ("canagliflozin", "FOURNIER"), ("canagliflozin", "KETOACIDOSIS"),
         ("dapagliflozin", "FOURNIER"),   ("dapagliflozin", "KETOACIDOSIS"),
         ("empagliflozin", "FOURNIER"),   ("empagliflozin", "KETOACIDOSIS"),
         ("liraglutide", "PANCREATITIS"), ("dulaglutide", "PANCREATITIS"), ("semaglutide", "PANCREATITIS")]

rows = []
for ing, rx in known:
    pool = sig[(sig.ingredient == ing) & (sig.a >= 3) & ~sig.is_noise]      # ranking pool for this drug
    hits = pool[pool.pt.str.contains(rx, regex=True)]
    for _, r in hits.iterrows():
        rows.append(dict(ingredient=ing, family=rx, pt=r.pt, a=int(r.a), IC025=round(r.IC025, 2),
                         EB05=round(r.EB05, 2), methods=int(r.n_methods),
                         rank_by_EB05=int((pool.EB05 > r.EB05).sum() + 1), of=len(pool)))
val = pd.DataFrame(rows).sort_values(["ingredient", "family", "rank_by_EB05"])
display(val)

# Sensitivity: share of known families with at least one fully flagged PT
fam = val.groupby(["ingredient", "family"]).methods.max().reset_index()
print("known families flagged by all four methods:", int((fam.methods == 4).sum()), "of", len(known))

# Lead times against FDA dates
display(con.sql("SELECT * FROM v_known_signal_family ORDER BY ingredient, pt_regex").df())

,ingredient,family,pt,a,IC025,EB05,methods,rank_by_EB05,of
0,canagliflozin,AMPUTATION,LIMB AMPUTATION,233,7.21,236.46,4,1,871
4,canagliflozin,AMPUTATION,TOE AMPUTATION,1915,7.55,199.23,4,5,871
2,canagliflozin,AMPUTATION,FOOT AMPUTATION,517,7.31,194.17,4,6,871
1,canagliflozin,AMPUTATION,AMPUTATION,285,6.91,160.97,4,9,871
3,canagliflozin,AMPUTATION,LEG AMPUTATION,668,6.89,133.30,4,12,871
6,canagliflozin,AMPUTATION,FINGER AMPUTATION,16,3.14,17.68,4,67,871
5,canagliflozin,AMPUTATION,HAND AMPUTATION,3,0.52,7.21,4,110,871
7,canagliflozin,FOURNIER,FOURNIER'S GANGRENE,126,6.08,103.00,4,20,871
9,canagliflozin,KETOACIDOSIS,DIABETIC KETOACIDOSIS,2589,6.95,127.99,4,14,871
8,canagliflozin,KETOACIDOSIS,KETOACIDOSIS,461,6.71,122.58,4,16,871


known families flagged by all four methods: 10 of 10


,ingredient,pt_regex,reg_date,first_pt,status,first_detect,lead_days
0,canagliflozin,AMPUTATION,2016-05-18,TOE AMPUTATION,emerging,2016-06-30,-43
1,canagliflozin,FOURNIER,2018-08-29,FOURNIER'S GANGRENE,emerging,2018-11-30,-93
2,canagliflozin,KETOACIDOSIS,2015-05-15,DIABETIC KETOACIDOSIS,pre-existing,2014-02-28,441
3,dapagliflozin,FOURNIER,2018-08-29,FOURNIER'S GANGRENE,emerging,2018-12-31,-124
4,dapagliflozin,KETOACIDOSIS,2015-05-15,DIABETIC KETOACIDOSIS,pre-existing,2014-09-30,227
5,dulaglutide,PANCREATITIS,NaT,PANCREATITIS,emerging,2015-05-31,<NA>
6,empagliflozin,FOURNIER,2018-08-29,FOURNIER'S GANGRENE,emerging,2018-10-31,-63
7,empagliflozin,KETOACIDOSIS,2015-05-15,DIABETIC KETOACIDOSIS,emerging,2015-02-28,76
8,liraglutide,PANCREATITIS,NaT,PANCREATITIS,pre-existing,2014-01-31,<NA>
9,semaglutide,PANCREATITIS,NaT,PANCREATITIS,emerging,2018-09-30,<NA>


#### 6: Sensitivity analyses

A signal is more convincing when it survives each of these changes. 

In [95]:
variants = {"main (a>=3)": flag(res, 3),
            "a>=5": flag(res, 5),
            "no noise PTs": flag(res[~res.is_noise], 3)}

def keyset(df): return set(zip(df[df.signal].ingredient, df[df.signal].pt))
base = keyset(variants["main (a>=3)"])
summ = pd.DataFrame({k: {"signals": len(keyset(v)),
                         "overlap_with_main": len(keyset(v) & base),
                         "jaccard": round(len(keyset(v) & base) / max(len(keyset(v) | base), 1), 2)}
                     for k, v in variants.items()}).T
display(summ)

# Pairs that were flagged by the monthly series at some point, but not signalling at end of 2019
ever = lab[lab.status != "never"].merge(sig[["ingredient", "pt", "signal"]], on=["ingredient", "pt"], how="left")
lost = ever[~ever.signal.fillna(False).astype(bool)]
print(len(ever), "ever signalled |", len(lost), "not signalling at end of 2019")
print(lost.groupby("ingredient").size())

,signals,overlap_with_main,jaccard
main (a>=3),955.0,955.0,1.00
a>=5,819.0,819.0,0.86
no noise PTs,835.0,835.0,0.87


1138 ever signalled | 183 not signalling at end of 2019
ingredient
canagliflozin    73
dapagliflozin    15
dulaglutide      31
empagliflozin    28
ertugliflozin     2
liraglutide      26
semaglutide       8
dtype: int64


#### Exporting the files for further analysis

In [97]:
import pathlib, datetime
import pandas as pd

out = pathlib.Path("outputs/signal_identification_v1")
out.mkdir(parents=True, exist_ok=True)

# 1. Full pair-level table (all 9,251 pairs, with flags, tier and is_noise)
sig.to_csv(out / "sig_final.csv", index=False)

# 2. Strong signals and watch list
sig[sig.tier == "1 Strong"].to_csv(out / "strong_signals.csv", index=False)
sig[(sig.tier == "2 Signal, not strong") & (sig.a >= 5)].to_csv(out / "watchlist.csv", index=False)

# 3. Raw metrics before flags (lets you change thresholds later)
res.to_csv(out / "res_raw.csv", index=False)

# 4. Settings and counts as a small CSV
nn = sig[~sig.is_noise]
settings = pd.DataFrame({
    "item": ["created", "min_a", "evans", "ror", "ic", "eb", "signal", "strong",
             "noise_regex", "n_pairs", "n_noise", "n_signal", "n_strong"],
    "value": [datetime.datetime.now().isoformat(timespec="seconds"), 3,
              "a>=3 & PRR>=2 & chi2>=4", "ROR_lo>1", "IC025>0", "EB05>=2",
              "evans & ror_sig & ic_sig", "signal & eb_sig",
              NOISE, len(sig), int(sig.is_noise.sum()),
              int(nn.signal.sum()), int(nn.strong.sum())]
})
settings.to_csv(out / "run_settings.csv", index=False)

# 5. Per-drug counts
(nn.groupby("ingredient")[["signal", "strong"]].sum()
   .to_csv(out / "counts_by_drug.csv"))

print(sorted(p.name for p in out.iterdir()))

['counts_by_drug.csv', 'res_raw.csv', 'run_settings.csv', 'sig_final.csv', 'strong_signals.csv', 'watchlist.csv']


### Close the DB connection

In [98]:
con.close()

# Explicitly close any ambient DuckDB connections
try:
    duckdb.close()
    print("✓ Closed ambient DuckDB connection.")
except Exception as e:
    print(f"Notice: {e}")

✓ Closed ambient DuckDB connection.
